# Séance 1 — Entraîner un U-Net pour la segmentation sémantique

*Cours « Compression de modèles pour l'IA embarquée » — séance 1/7 — 3 h*

Avant de compresser un modèle, il faut un modèle. Aujourd'hui, vous construisez et entraînez le U-Net qui servira de **référence** pendant tout le cours : export ONNX, quantification, élagage et distillation seront tous mesurés par rapport à ce que vous produisez dans ce notebook.

**À la fin de la séance, vous saurez :**
- expliquer l'architecture U-Net et le rôle des connexions de saut (*skip connections*) ;
- écrire un U-Net dont la largeur est un paramètre ;
- l'entraîner sur CamVid et l'évaluer avec le mIoU, classe par classe ;
- dire où se trouvent les paramètres du réseau, et où se trouve le calcul (ce n'est pas au même endroit).

| Partie | Contenu | Durée |
|---|---|---|
| 0 | Mise en place | 5 min |
| 1 | Le problème et les données | 30 min |
| 2 | Construire le U-Net | 45 min |
| 3 | Fonction de coût et métrique | 25 min |
| 4 | Entraînement | 40 min |
| 5 | Analyse des résultats | 20 min |
| 6 | Premier compromis taille / précision | 15 min |

**Mode d'emploi.** Chaque partie commence par une courte introduction, puis c'est à vous. Chaque cellule de code porte une étiquette sur sa première ligne :

| Étiquette | Ce que vous faites |
|---|---|
| `# ✏️ À COMPLÉTER` | Vous écrivez du code : chaque `A_COMPLETER()` est à remplacer, en suivant la consigne placée juste au-dessus. |
| `# ✅ VÉRIFICATION` | Vous exécutez sans modifier : la cellule vous dit si vous pouvez continuer. |
| `# ▶ FOURNI` | Vous exécutez sans modifier. |

Si vous oubliez un `A_COMPLETER()`, l'exécution s'arrête avec un message qui vous le signale. Les **Questions** se discutent avec votre voisin avant d'en parler ensemble.

## 0. Mise en place

Activez le GPU : *Exécution → Modifier le type d'exécution → GPU T4*. Le GPU sert uniquement à l'entraînement ; à partir de la séance 2, toutes les mesures se feront sur CPU.

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
import copy, glob, os, random, time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from matplotlib.patches import Patch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", device)

def A_COMPLETER():
    """Marque le code que vous devez écrire. À remplacer partout où il apparaît."""
    raise NotImplementedError("✏️ Code à compléter : remplacez A_COMPLETER() par votre code "
                              "(la consigne est juste au-dessus de la ligne signalée).")

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
# CamVid, version à 11 classes distribuée avec le tutoriel SegNet (environ 180 Mo)
if not os.path.exists("SegNet-Tutorial"):
    !git clone --depth 1 https://github.com/alexgkendall/SegNet-Tutorial.git

ROOT = "SegNet-Tutorial/CamVid"
for split in ["train", "val", "test"]:
    print(split, ":", len(os.listdir(f"{ROOT}/{split}")), "images")

## 1. Le problème et les données

**Segmentation sémantique** : attribuer une classe à *chaque pixel*. Le réseau reçoit une image `3 × H × W` et produit `C × H × W` scores, un par classe et par pixel.

**CamVid** : des scènes de conduite filmées depuis une voiture, annotées en 11 classes. C'est un cas typique d'IA embarquée : le modèle doit tourner dans le véhicule, en temps réel, sur un calculateur limité. Le jeu est petit (367 images d'entraînement, 101 de validation, 233 de test), ce qui permet d'entraîner en quelques minutes.

Dans les masques, chaque pixel vaut un entier de 0 à 10 (la classe). La valeur 11 signifie « non annoté » : ces pixels seront ignorés partout, dans la fonction de coût comme dans la métrique.

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
CLASSES = ["ciel", "bâtiment", "poteau", "route", "trottoir", "végétation",
           "panneau", "clôture", "voiture", "piéton", "cycliste"]
N_CLASSES = len(CLASSES)   # 11
IGNORE = 11                # étiquette « non annoté »
H, W = 240, 320            # résolution de travail pour tout le cours

# Une couleur par classe (la dernière, noire, pour « non annoté »)
PALETTE = np.array([[128, 128, 128], [128, 0, 0], [192, 192, 128], [128, 64, 128],
                    [60, 40, 222], [128, 128, 0], [192, 128, 128], [64, 64, 128],
                    [64, 0, 128], [64, 64, 0], [0, 128, 192], [0, 0, 0]], dtype=np.uint8)

### Exercice 1.1 — Charger les images

Les images d'origine font 360 × 480. Nous travaillons en **240 × 320** : l'entraînement est plus rapide et l'inférence sur CPU restera raisonnable dans les séances suivantes. Les deux dimensions sont des multiples de 16, ce qui est nécessaire pour un U-Net à 4 sous-échantillonnages (vous verrez pourquoi en partie 2).

Complétez `load_split` : il manque le redimensionnement du masque.

**Question.** L'image est redimensionnée en bilinéaire. Pourquoi ne peut-on pas faire la même chose pour le masque ?

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
def load_split(split):
    """Charge un split ("train", "val" ou "test") en mémoire.

    Retourne les images en uint8 (N, 3, H, W) et les masques en uint8 (N, H, W)."""
    paths = sorted(glob.glob(f"{ROOT}/{split}/*.png"))
    images, masks = [], []
    for p in paths:
        img = Image.open(p).convert("RGB").resize((W, H), Image.BILINEAR)
        mask = Image.open(p.replace(f"/{split}/", f"/{split}annot/"))
        # ✏️ À COMPLÉTER : redimensionner le masque en (W, H) avec la bonne interpolation
        mask = A_COMPLETER()
        images.append(np.array(img))
        masks.append(np.array(mask))
    images = torch.from_numpy(np.stack(images)).permute(0, 3, 1, 2).contiguous()
    masks = torch.from_numpy(np.stack(masks))
    return images, masks


train_images, train_masks = load_split("train")
val_images, val_masks = load_split("val")
test_images, test_masks = load_split("test")

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
assert train_images.shape == (367, 3, H, W), train_images.shape
assert train_masks.shape == (367, H, W), train_masks.shape
_p = sorted(glob.glob(f"{ROOT}/train/*.png"))[0].replace("/train/", "/trainannot/")
_orig = set(np.unique(np.array(Image.open(_p))).tolist())
assert set(train_masks[0].unique().tolist()) <= _orig, \
    "Le redimensionnement a créé des classes absentes du masque d'origine : mauvaise interpolation."
print("OK :", train_images.shape, train_masks.shape)

### Exercice 1.2 — Regarder les données

Toujours regarder les données avant d'entraîner. La fonction `show` est fournie : utilisez-la pour afficher quelques images d'entraînement avec leur masque.

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
def show(images, masks, preds=None, n=3):
    """Affiche n images, leur masque et, si elle est fournie, la prédiction."""
    cols = 2 if preds is None else 3
    fig, axes = plt.subplots(n, cols, figsize=(4.5 * cols, 3.4 * n), squeeze=False)
    titles = ["Image", "Vérité terrain", "Prédiction"]
    for i in range(n):
        axes[i, 0].imshow(images[i].permute(1, 2, 0).numpy())
        axes[i, 1].imshow(PALETTE[masks[i].numpy()])
        if preds is not None:
            axes[i, 2].imshow(PALETTE[preds[i].numpy()])
        for j in range(cols):
            axes[i, j].set_title(titles[j]); axes[i, j].axis("off")
    legend = [Patch(color=PALETTE[c] / 255, label=CLASSES[c]) for c in range(N_CLASSES)]
    fig.legend(handles=legend, loc="lower center", ncol=6, frameon=False)
    plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : afficher 3 images d'entraînement et leur masque
A_COMPLETER()

### Exercice 1.3 — Toutes les classes ne pèsent pas le même poids

Calculez `freq`, un tenseur de 11 valeurs : la proportion de pixels de chaque classe dans les masques d'entraînement, sans compter les pixels « non annoté ».

Indication : `torch.bincount`.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : fréquence de chaque classe (tenseur float de taille 11, de somme 1)
freq = A_COMPLETER()

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification, puis affichage
assert freq.shape == (N_CLASSES,) and abs(freq.sum().item() - 1) < 1e-4

order = freq.argsort(descending=True)
plt.figure(figsize=(8, 3))
plt.bar([CLASSES[i] for i in order], 100 * freq[order], color=[PALETTE[i] / 255 for i in order])
plt.ylabel("% des pixels"); plt.xticks(rotation=45, ha="right"); plt.show()
print(f"Classe la plus fréquente / la plus rare : {freq.max() / freq.min():.0f} fois plus de pixels")

**Question.** Un modèle qui ne prédirait jamais « poteau », « panneau », « piéton » ni « cycliste » aurait quel pourcentage de pixels corrects, au mieux ? Qu'en concluez-vous sur le choix de la métrique ?

### Exercice 1.4 — Dataset et augmentation

Avec 367 images, le modèle apprend vite par cœur. On limite ce sur-apprentissage par deux augmentations simples : un **retournement horizontal** aléatoire et une variation de luminosité (fournie).

Complétez le retournement. Attention à ce qui doit être retourné.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class CamVid(Dataset):
    def __init__(self, images, masks, augment=False):
        self.images, self.masks, self.augment = images, masks, augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        x = self.images[i].float() / 255      # (3, H, W) dans [0, 1]
        y = self.masks[i].long()              # (H, W)
        if self.augment:
            if random.random() < 0.5:
                # ✏️ À COMPLÉTER : retournement horizontal (torch.flip)
                A_COMPLETER()
            x = (x * random.uniform(0.8, 1.2)).clamp(0, 1)   # luminosité
        return (x - MEAN) / STD, y


train_ds = CamVid(train_images, train_masks, augment=True)
val_ds = CamVid(val_images, val_masks)
test_ds = CamVid(test_images, test_masks)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification : sur plusieurs tirages, on doit voir l'image retournée ET le masque retourné ensemble
_seen = set()
for _ in range(40):
    _x, _y = train_ds[0]
    _flip_y = torch.equal(_y, torch.flip(train_masks[0].long(), dims=[1]))
    _same_y = torch.equal(_y, train_masks[0].long())
    _ref = train_images[0].float()
    _flip_x = (_x * STD + MEAN - torch.flip(_ref, dims=[2]) / 255).abs().mean() < \
              (_x * STD + MEAN - _ref / 255).abs().mean()
    assert _flip_y or _same_y
    assert bool(_flip_x) == _flip_y, "L'image et le masque ne sont pas retournés ensemble."
    _seen.add(_flip_y)
assert _seen == {True, False}, "Le retournement n'a jamais lieu (ou a toujours lieu)."
x, y = next(iter(train_loader))
print("OK :", x.shape, x.dtype, "|", y.shape, y.dtype)

## 2. Construire le U-Net

Trois idées suffisent pour comprendre l'architecture :

1. **L'encodeur** réduit la résolution par 2 à chaque étage et double le nombre de canaux. Il perd le « où » et gagne le « quoi ».
2. **Le décodeur** fait le chemin inverse pour revenir à la résolution de l'image.
3. **Les connexions de saut** recopient les cartes de l'encodeur vers l'étage du décodeur de même résolution. Elles rapportent les détails fins (bords, objets minces) que le sous-échantillonnage a détruits.

```
image 3×240×320
   │
 enc1 (32) ──────────────────────────────────► dec1 (32) ──► tête 1×1 ──► 11×240×320
   │ pool                                         ▲ up
 enc2 (64) ───────────────────────────► dec2 (64)
   │ pool                                  ▲ up
 enc3 (128) ──────────────────► dec3 (128)
   │ pool                          ▲ up
 enc4 (256) ──────────► dec4 (256)
   │ pool                  ▲ up
   └──► bottleneck (512) ──┘
```

Les nombres entre parenthèses sont les canaux pour une largeur de base de 32. **Cette largeur sera un paramètre** du modèle : nous la réutiliserons pour l'élagage et la distillation.

### Exercice 2.1 — Le bloc de base

Chaque étage est un bloc `DoubleConv` : deux fois de suite *convolution 3×3 → BatchNorm → ReLU*. Les convolutions utilisent `padding=1` (la résolution ne change pas) et `bias=False`.

**Question.** Pourquoi peut-on supprimer le biais d'une convolution suivie d'une BatchNorm ? Gardez votre réponse en tête : elle sera le point de départ de la séance 2.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
class DoubleConv(nn.Module):
    """(Conv 3×3 → BatchNorm → ReLU) × 2"""

    def __init__(self, c_in, c_out):
        super().__init__()
        # ✏️ À COMPLÉTER : construire le bloc avec nn.Sequential
        self.block = A_COMPLETER()

    def forward(self, x):
        return self.block(x)

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
_b = DoubleConv(3, 32)
assert _b(torch.randn(2, 3, 64, 80)).shape == (2, 32, 64, 80), "La résolution ne doit pas changer."
assert sum(p.numel() for p in _b.parameters()) == 10_208, "Vérifiez bias=False et les deux BatchNorm."
print("OK")

### Exercice 2.2 — Assembler le réseau

L'encodeur est fourni. Écrivez le décodeur, puis la passe avant.

À chaque étage du décodeur :
- `nn.ConvTranspose2d(c_in, c_out, kernel_size=2, stride=2)` double la résolution et divise les canaux par 2 ;
- on **concatène** le résultat avec la sortie de l'encodeur de même résolution (`torch.cat(..., dim=1)`) ;
- un `DoubleConv` fusionne le tout. Attention à son nombre de canaux d'entrée après la concaténation.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
class UNet(nn.Module):
    def __init__(self, n_classes=N_CLASSES, base=32):
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]

        # Encodeur
        self.enc1 = DoubleConv(3, c[0])
        self.enc2 = DoubleConv(c[0], c[1])
        self.enc3 = DoubleConv(c[1], c[2])
        self.enc4 = DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c[3], c[4])

        # Décodeur
        self.up4 = nn.ConvTranspose2d(c[4], c[3], kernel_size=2, stride=2)
        self.dec4 = DoubleConv(c[4], c[3])     # c[4] = c[3] (up4) + c[3] (enc4)
        # ✏️ À COMPLÉTER : up3/dec3, up2/dec2, up1/dec1 sur le même modèle
        A_COMPLETER()

        # Tête : une convolution 1×1 qui produit un score par classe
        self.head = nn.Conv2d(c[0], n_classes, kernel_size=1)

    def forward(self, x):
        # ✏️ À COMPLÉTER : encodeur (gardez e1, e2, e3, e4 pour les connexions de saut)
        A_COMPLETER()
        # ✏️ À COMPLÉTER : bottleneck
        A_COMPLETER()
        # ✏️ À COMPLÉTER : décodeur avec concaténations
        A_COMPLETER()
        return self.head(d1)

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
model = UNet(N_CLASSES, base=32)
out = model(torch.randn(2, 3, H, W))
assert out.shape == (2, N_CLASSES, H, W), out.shape
assert sum(p.numel() for p in model.parameters()) == 7_763_371, "Nombre de paramètres inattendu."
print("OK :", out.shape)

**Question.** Essayez `model(torch.randn(1, 3, 250, 320))`. Que se passe-t-il, et pourquoi la hauteur et la largeur doivent-elles être des multiples de 16 ?

### Exercice 2.3 — Où sont les paramètres ? Où est le calcul ?

Écrivez `count_params`, puis exécutez la cellule suivante, qui détaille le réseau bloc par bloc.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
def count_params(module):
    """Nombre total de paramètres d'un module."""
    # ✏️ À COMPLÉTER : écrire le corps de la fonction
    A_COMPLETER()

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
total = count_params(model)
print(f"Total : {total / 1e6:.2f} M paramètres, soit {total * 4 / 1e6:.0f} Mo en float32\n")

# Taille de la sortie de chaque bloc pour une image 240×320
sizes = {}
hooks = [m.register_forward_hook(lambda mod, inp, out, name=name: sizes.update({name: out.shape}))
         for name, m in model.named_children() if name != "pool"]
with torch.no_grad():
    model.eval()(torch.randn(1, 3, H, W))
for h in hooks:
    h.remove()

print(f"{'bloc':12s}{'paramètres':>12s}{'% du total':>12s}   {'sortie (C×H×W)':18s}{'valeurs en sortie':>18s}")
for name, m in model.named_children():
    if name == "pool":
        continue
    n, s = count_params(m), sizes[name]
    print(f"{name:12s}{n:>12,d}{100 * n / total:>11.1f}%   {str(tuple(s[1:])):18s}{s[1:].numel():>18,d}")

**Questions.**
1. Quels blocs concentrent la majorité des paramètres ?
2. Quels blocs produisent les plus grosses cartes d'activation, donc demandent le plus de mémoire vive pendant l'inférence ?
3. Si votre cible embarquée manque de **stockage**, quels blocs faut-il réduire en priorité ? Et si elle manque de **mémoire vive** ?

## 3. Fonction de coût et métrique

**Fonction de coût.** La segmentation est une classification pixel par pixel : on utilise une entropie croisée, moyennée sur les pixels. Deux réglages :
- `ignore_index=IGNORE` pour que les pixels non annotés ne comptent pas ;
- un **poids par classe**, plus grand pour les classes rares, sinon la route et le ciel dominent l'apprentissage. Nous prenons $w_c = 1 / \ln(1{,}02 + f_c)$, où $f_c$ est la fréquence de la classe.

### Exercice 3.1 — Poids et fonction de coût

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : poids des classes à partir de freq (exercice 1.3)
class_weights = A_COMPLETER()

# ✏️ À COMPLÉTER : entropie croisée pondérée qui ignore les pixels non annotés
criterion = A_COMPLETER()

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification
for c in freq.argsort():
    print(f"{CLASSES[c]:12s} fréquence {100 * freq[c]:5.2f} %   poids {class_weights[c]:5.1f}")
assert class_weights.shape == (N_CLASSES,)
assert class_weights[freq.argmin()] == class_weights.max(), "La classe la plus rare doit avoir le plus grand poids."
_l = criterion(torch.zeros(1, N_CLASSES, 2, 2, device=device), torch.tensor([[[0, 1], [2, IGNORE]]], device=device))
assert abs(_l.item() - np.log(N_CLASSES)) < 1e-4
print("OK")

### Exercice 3.2 — Le mIoU

Pour une classe, l'**IoU** (*intersection over union*) compare les pixels prédits dans cette classe et ceux qui lui appartiennent vraiment :

$$\text{IoU}_c = \frac{\text{pixels bien prédits de la classe } c}{\text{pixels prédits } c \;\cup\; \text{pixels réellement } c}$$

Le **mIoU** est la moyenne des IoU sur les classes : un poteau compte autant que la route.

Tout se calcule à partir de la **matrice de confusion** `cm`, où `cm[i, j]` est le nombre de pixels de vraie classe `i` prédits en classe `j`. On accumule cette matrice sur tout le jeu d'évaluation, puis on calcule les IoU une seule fois à la fin.

Indication pour la matrice : chaque couple (vraie classe, classe prédite) peut être codé par un seul entier `vraie * n_classes + prédite`, puis compté avec `torch.bincount`.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
def confusion_matrix(pred, target, n_classes=N_CLASSES, ignore=IGNORE):
    """pred et target : tenseurs d'entiers de même forme. Retourne cm (n_classes, n_classes)."""
    # ✏️ À COMPLÉTER : écarter les pixels ignorés, puis compter les couples (vraie classe, classe prédite)
    A_COMPLETER()


def iou_per_class(cm):
    """IoU de chaque classe à partir de la matrice de confusion."""
    # ✏️ À COMPLÉTER : intersection = diagonale ; union = total ligne + total colonne - diagonale
    A_COMPLETER()

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification sur un exemple calculable à la main
_t = torch.tensor([0, 0, 1, 1, IGNORE])
_p = torch.tensor([0, 1, 1, 1, 0])
_cm = confusion_matrix(_p, _t, n_classes=2)
assert _cm.tolist() == [[1, 1], [0, 2]], _cm
assert torch.allclose(iou_per_class(_cm), torch.tensor([1 / 2, 2 / 3]))
print("OK")

## 4. Entraînement

Rien de spécifique à la segmentation ici : c'est la boucle PyTorch habituelle. Vous écrivez l'entraînement sur une époque et l'évaluation ; la fonction `fit`, fournie, enchaîne les époques et conserve les poids de la meilleure époque selon le mIoU de validation.

### Exercice 4.1 — Une époque d'entraînement, une évaluation

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
def train_one_epoch(model, loader, criterion, optimizer, scheduler):
    """Entraîne le modèle sur une époque. Retourne la valeur moyenne de la fonction de coût."""
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # ✏️ À COMPLÉTER : remise à zéro des gradients, passe avant, coût, rétropropagation, mise à jour
        A_COMPLETER()
        scheduler.step()                      # le pas d'apprentissage change à chaque itération
        total += loss.item() * x.size(0)
    return total / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Retourne (coût moyen, mIoU, IoU par classe) sur un jeu de données."""
    model.eval()
    total = 0.0
    cm = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long, device=device)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # ✏️ À COMPLÉTER : passe avant, coût, classe prédite par pixel, accumulation dans cm
        A_COMPLETER()
        total += loss.item() * x.size(0)
    iou = iou_per_class(cm).cpu()
    return total / len(loader.dataset), iou.mean().item(), iou

In [ ]:
# ✅ VÉRIFICATION — à exécuter sans modifier
# Vérification : un réseau non entraîné doit être proche du hasard
_loss, _miou, _ = evaluate(UNet(N_CLASSES, base=8).to(device), val_loader, criterion)
print(f"Avant entraînement : coût {_loss:.2f} (hasard : {np.log(N_CLASSES):.2f}), mIoU {100 * _miou:.1f} %")
assert 1.5 < _loss < 4 and _miou < 0.15
print("OK")

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
def fit(model, epochs, lr=1e-3):
    """Entraîne le modèle et le laisse dans l'état de sa meilleure époque (mIoU de validation)."""
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr,
                                                    total_steps=epochs * len(train_loader))
    history = {"train_loss": [], "val_loss": [], "val_miou": []}
    best_miou, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scheduler)
        val_loss, val_miou, _ = evaluate(model, val_loader, criterion)
        for k, v in zip(history, (train_loss, val_loss, val_miou)):
            history[k].append(v)
        if val_miou > best_miou:
            best_miou, best_state = val_miou, copy.deepcopy(model.state_dict())
        print(f"époque {epoch:2d}/{epochs} | coût train {train_loss:.3f} | coût val {val_loss:.3f} "
              f"| mIoU val {100 * val_miou:5.1f} % | {time.time() - t0:.0f} s")
    model.load_state_dict(best_state)
    print(f"Meilleur mIoU de validation : {100 * best_miou:.1f} %")
    return history

### Exercice 4.2 — Lancer l'entraînement

Sur le GPU de Colab, comptez quelques minutes. Pendant ce temps, observez les trois colonnes : le coût d'entraînement, le coût de validation et le mIoU évoluent-ils ensemble ?

Ordre de grandeur attendu : un mIoU de validation de 50 à 65 %. En dessous de 30 % après 40 époques, cherchez une erreur (masques mal redimensionnés, retournement appliqué à l'image seule, concaténation dans le mauvais ordre).

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
EPOCHS = 40

model = UNet(N_CLASSES, base=32)
history = fit(model, epochs=EPOCHS)

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.5))
a.plot(history["train_loss"], label="entraînement"); a.plot(history["val_loss"], label="validation")
a.set_xlabel("époque"); a.set_ylabel("coût"); a.legend()
b.plot([100 * v for v in history["val_miou"]]); b.set_xlabel("époque"); b.set_ylabel("mIoU de validation (%)")
plt.show()

**Question.** Voyez-vous du sur-apprentissage sur les courbes ? À quoi le reconnaît-on, et pourquoi `fit` conserve-t-elle la meilleure époque plutôt que la dernière ?

## 5. Analyse des résultats

Le jeu de validation a servi à choisir la meilleure époque. Le chiffre que l'on rapporte est donc celui du **jeu de test**, que le modèle n'a jamais vu.

### Exercice 5.1 — IoU par classe sur le test

Évaluez le modèle sur le jeu de test et affichez l'IoU de chaque classe à côté de sa fréquence. Ce tableau est votre **référence** : gardez-le, vous le comparerez à chaque version compressée du modèle.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : évaluer sur le jeu de test
test_loss, test_miou, test_iou = A_COMPLETER()

# ✏️ À COMPLÉTER : afficher, pour chaque classe, son IoU et sa fréquence, puis le mIoU
A_COMPLETER()

### Exercice 5.2 — Regarder les prédictions

Calculez les prédictions du modèle pour les 3 premières images de test et affichez-les avec `show`.

Indication : `test_ds[i]` renvoie l'image normalisée attendue par le modèle ; `test_images[i]` est l'image d'origine, pour l'affichage.

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : prédictions (tenseur d'entiers de taille (3, H, W), sur CPU), puis affichage
preds = A_COMPLETER()

**Questions.**
1. Quelles classes sont mal segmentées ? Reliez vos observations à la fréquence des classes et à la taille des objets dans l'image.
2. Sur les images, où se trouvent les erreurs : à l'intérieur des objets ou sur leurs bords ?
3. Avec 233 images de test, un écart de 0,5 point de mIoU entre deux modèles vous semble-t-il significatif ? Comment le vérifier ?

## 6. Premier compromis taille / précision

La façon la plus simple de réduire un modèle est de l'entraîner plus petit. Ce n'est pas encore de la compression, mais c'est le point de comparaison de toutes les techniques du cours : une méthode de compression n'a d'intérêt que si elle fait mieux qu'un petit modèle entraîné directement.

### Exercice 6.1 — Un U-Net deux fois moins large

Entraînez un U-Net avec `base=16` dans les mêmes conditions et évaluez-le sur le jeu de test.

**Avant de lancer** : par combien le nombre de paramètres sera-t-il divisé ? Et le temps d'inférence ?

In [ ]:
# ✏️ À COMPLÉTER — remplacez chaque A_COMPLETER() par votre code
# ✏️ À COMPLÉTER : créer model16, l'entraîner, calculer son mIoU de test (test_miou16)
A_COMPLETER()

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
@torch.no_grad()
def cpu_latency_ms(model, n=10):
    """Temps moyen d'une inférence sur CPU, pour une image, en millisecondes."""
    m = copy.deepcopy(model).cpu().eval()
    x = torch.randn(1, 3, H, W)
    for _ in range(2):          # échauffement
        m(x)
    t0 = time.perf_counter()
    for _ in range(n):
        m(x)
    return 1000 * (time.perf_counter() - t0) / n


print(f"{'modèle':10s}{'paramètres':>12s}{'mIoU test':>12s}{'latence CPU':>14s}")
for name, m, miou in [("base=32", model, test_miou), ("base=16", model16, test_miou16)]:
    print(f"{name:10s}{count_params(m) / 1e6:>10.2f} M{100 * miou:>10.1f} %{cpu_latency_ms(m):>11.0f} ms")

**Questions.**
1. Le nombre de paramètres est divisé par 4. La latence aussi ? Le mIoU ?
2. Quelles classes perdent le plus entre `base=32` et `base=16` ? (Comparez les IoU par classe.)

La mesure de latence ci-dessus est grossière : PyTorch, sur le CPU partagé de Colab. En séance 2, nous ferons des mesures propres avec ONNX Runtime.

## 7. Sauvegarde

Le modèle `base=32` est votre référence pour la suite. La cellule ci-dessous l'enregistre avec tout ce qu'il faut pour le recharger en séance 2. **Téléchargez le fichier** (ou copiez-le sur votre Drive) : l'espace de travail Colab est effacé à la fin de la session.

In [ ]:
# ▶ FOURNI — à exécuter sans modifier
checkpoint = {
    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
    "base": 32,
    "n_classes": N_CLASSES,
    "classes": CLASSES,
    "input_size": (H, W),
    "test_miou": test_miou,
    "test_iou": test_iou.tolist(),
}
torch.save(checkpoint, "unet_camvid_base32.pt")
print(f"Enregistré : unet_camvid_base32.pt ({os.path.getsize('unet_camvid_base32.pt') / 1e6:.0f} Mo)")

# Dans Colab, pour télécharger le fichier :
# from google.colab import files; files.download("unet_camvid_base32.pt")

## À retenir

- Un U-Net, c'est un encodeur, un décodeur et des connexions de saut qui rapportent les détails fins.
- La segmentation s'évalue au **mIoU**, classe par classe : le taux de pixels corrects cache l'échec sur les petites classes.
- **Les paramètres sont dans les couches profondes, les grosses activations dans les couches de haute résolution.** Taille du fichier, mémoire vive et temps de calcul sont trois contraintes différentes.
- Un modèle deux fois moins large a quatre fois moins de paramètres et perd peu : les réseaux sont redondants. Toute technique de compression devra faire mieux que ce point de comparaison.

## Pour la séance 2

Nous sortirons de PyTorch : export du modèle en **ONNX**, inspection du graphe, mesures de latence et de mémoire sur CPU avec ONNX Runtime, et première optimisation sans aucune perte, la fusion convolution + BatchNorm.

D'ici là : conservez votre fichier `unet_camvid_base32.pt` et votre tableau d'IoU par classe.